In [1]:
import pandas as pd
import numpy as np
from execution.backtest_engine import BacktestEngine

In [2]:
# 1. Generujemy sztuczne ceny otwarcia (6 kolejnych dni)
dates = pd.date_range(start="2024-01-01", periods=6, freq="D")

df_open_prices = pd.DataFrame({
    'PEP': [100.0, 102.0, 105.0, 103.0, 100.0, 101.0], # Ceny dla Y
    'KO':  [50.0,  51.0,  52.0,  51.0,  50.0,  49.0]   # Ceny dla X
}, index=dates)

print("\n1. Wygenerowane ceny otwarcia na giełdzie:")
print(df_open_prices)

# 2. Generujemy sztuczne sygnały z SignalGeneratora
trades = [
    {
        'pozycja': 'long',
        'entry_date': pd.Timestamp('2024-01-01'),
        'close_date': pd.Timestamp('2024-01-03')
    },
    {
        'pozycja': 'short',
        'entry_date': pd.Timestamp('2024-01-04'),
        'close_date': pd.Timestamp('2024-01-05')
    }
]


1. Wygenerowane ceny otwarcia na giełdzie:
              PEP    KO
2024-01-01  100.0  50.0
2024-01-02  102.0  51.0
2024-01-03  105.0  52.0
2024-01-04  103.0  51.0
2024-01-05  100.0  50.0
2024-01-06  101.0  49.0


In [3]:
engine = BacktestEngine(
    initial_capital=100_000.0,
    allocation_per_trade=0.10,
    commission_bps=3.0,
    slippage_bps=3.0,
    borrow_rate_annual=0.01
)

In [4]:
pair = ('PEP', 'KO')
beta_test = 0.5 # Zakładamy dodatnią betę

In [5]:
# --- KROK 1: MAPOWANIE CEN ---
print("\n--- KROK 1: MAPOWANIE CEN (match_execution_prices) ---")
df_matched = engine.match_execution_prices(trades, df_open_prices, pair)
print(df_matched[['pozycja', 'entry_date', 'close_date', 'entry_y', 'exit_y', 'entry_x', 'exit_x']])


--- KROK 1: MAPOWANIE CEN (match_execution_prices) ---
  pozycja entry_date close_date  entry_y  exit_y  entry_x  exit_x
0    long 2024-01-01 2024-01-03    102.0   103.0     51.0    51.0
1   short 2024-01-04 2024-01-05    100.0   101.0     50.0    49.0


In [6]:
# --- KROK 2: KALKULACJA PNL ---
print(f"\n--- KROK 2: KALKULACJA ZYSKÓW I KOSZTÓW (calculate_trades_pnl) | Beta = {beta_test} ---")
df_pnl = engine.calculate_trades_pnl(df_matched, beta=beta_test)
print(df_pnl[['pozycja', 'gross_pnl', 'execution_cost', 'borrow_cost', 'net_pnl', 'net_return_pct']])


--- KROK 2: KALKULACJA ZYSKÓW I KOSZTÓW (calculate_trades_pnl) | Beta = 0.5 ---
  pozycja   gross_pnl  execution_cost  borrow_cost     net_pnl  net_return_pct
0    long   65.359477       12.039216     0.182648   53.137613        0.005314
1   short -133.333333       12.000000     0.182648 -145.515982       -0.014552


In [8]:
# --- KROK 3: STATYSTYKI PORTFELA ---
print("\n--- KROK 3: METRYKI PORTFELOWE (calculate_performance_metrics) ---")
metrics = engine.calculate_performance_metrics(df_pnl)
for key, value in metrics.items():
    print(f"{key}: {value}")


--- KROK 3: METRYKI PORTFELOWE (calculate_performance_metrics) ---
total_trades: 2
total_pnl: -92.38
total_return_pct: -0.09
win_rate_pct: 50.0
profit_factor: 0.37
max_drawdown_pct: 0.15
sharpe_ratio: -4.44


In [ ]:
for key, value in metrics.items():
    print(f"{key}: {value}")